In [10]:
import numpy as np
import keras
from keras import layers
import os
import cv2
import matplotlib.pyplot as plt


In [11]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("wanderdust/coin-images")

print("Path to dataset files:", path)

C:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\USER\.cache\kagglehub\datasets\wanderdust\coin-images\versions\1


In [12]:
inputs = keras.Input(shape=(784,))

In [13]:
dense = layers.Dense(64, activation="relu")
x = dense(inputs)

In [14]:
x = layers.Dense(64, activation="relu")(x)
outputs = layers.Dense(10)(x)

In [15]:
dataset_path = r"C:\Users\USER\.cache\kagglehub\datasets\wanderdust\coin-images\versions\1"

for root, dirs, files in os.walk(dataset_path):
    depth = root.replace(dataset_path, "").count(os.sep)
    if depth <= 3:
        print(f"{'  ' * depth}{os.path.basename(root)}/  ({len(files)} files)")

1/  (1 files)
  coins/  (0 files)
    data/  (1 files)
      test/  (0 files)
      train/  (0 files)
      validation/  (0 files)


In [16]:
import os, json, glob

json_path = glob.glob(os.path.join(dataset_path, "**", "cat_to_name.json"), recursive=True)
print(json_path)

with open(json_path[0]) as f:
    cat_to_name = json.load(f)

['C:\\Users\\USER\\.cache\\kagglehub\\datasets\\wanderdust\\coin-images\\versions\\1\\cat_to_name.json', 'C:\\Users\\USER\\.cache\\kagglehub\\datasets\\wanderdust\\coin-images\\versions\\1\\coins\\data\\cat_to_name.json']


In [17]:
data_dir = os.path.join(dataset_path, "coins", "data")
train_dir = os.path.join(data_dir, "train")
val_dir   = os.path.join(data_dir, "validation")
test_dir  = os.path.join(data_dir, "test")

classes = sorted(os.listdir(train_dir))
print(len(classes), classes[:5])
print(len(os.listdir(os.path.join(train_dir, classes[0]))), "images in first class")

211 ['1', '10', '100', '101', '102']
28 images in first class


In [19]:
import os
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Adjust if your structure differs (e.g. there may be an extra "data" folder)
data_dir = os.path.join(dataset_path, "coins", "data")
train_dir = os.path.join(data_dir, "train")
val_dir   = os.path.join(data_dir, "validation")   # may be "valid"
test_dir  = os.path.join(data_dir, "test")

IMG_SIZE = 224
BATCH_SIZE = 32 #how mant images go through the model at finetuning. How many are we looking at before updating the weights,
#batch size is to look at when were looking at training time and resources

# ImageNet stats - use these if you plan to use a pretrained model
#Noramlise computes (pixel - mean) / std
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

#rotate, flip, and color jitter make the image look a bit different every time to make the model learn what a coin looks like, rrather than what the images look like
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(20),          # coins can be any orientation
    transforms.RandomHorizontalFlip(), # play with this. coins dont flip
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), #converts pixels from 0-255 to 0-1 and rearranged image into a tensor
    transforms.Normalize(mean, std), #shifts and scales the tensor
])

#we don't need random augmentation for the evaluation and testing sets
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
#the transforms are stored in the ImageFolder and applied each time the image is loaded. This ensures the random augmentations are different every epoch
train_ds = datasets.ImageFolder(train_dir, transform=train_transform)
val_ds   = datasets.ImageFolder(val_dir,   transform=eval_transform)
test_ds  = datasets.ImageFolder(test_dir,  transform=eval_transform)

#wraps a dataset and hands it off to the model, armed with all the level settings we decided on
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("Classes:", len(train_ds.classes)) #counts number of classes
print("Train/Val/Test sizes:", len(train_ds), len(val_ds), len(test_ds))

Classes: 211
Train/Val/Test sizes: 6413 844 844


In [20]:
from collections import Counter
counts = Counter(train_ds.targets)
print(min(counts.values()), max(counts.values()))

8 52


In [21]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cpu


In [22]:
import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(36)

In [25]:
import torch.nn as nn
from torchvision import models

num_classes = len(train_ds.classes)

#resnet is pretrained on imagenet, so it knows to recognize features of an image
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

for p in model.parameters():
    #requires_grad flags each parameter on whether or not to compute gradients for it.
    #were setting it to false for now so that they're excluded from learning to keep pretrained knowledge in tact and save cpu
    #consider playing with it
    p.requires_grad = False          # freeze the pretrained backbone

#model fc (fully connected) is the last layer of resnet, turns the features into class scores
#this is the only part that learns
model.fc = nn.Linear(model.fc.in_features, num_classes)   # new head for 211 coins
model = model.to(device)

#how wrong is the model? 
criterion = nn.CrossEntropyLoss()
#tries to minimize the wrongness
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

In [28]:
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, n = 0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader: #x - images, a tensor shaped [32, 3, 224, 224], y - true labels, tensor or 32 class indices
            x, y = x.to(device), y.to(device) #both sent to device
            out = model(x)
            loss = criterion(out, y) #average loss per image in batch
            if train:
                optimizer.zero_grad() #clear the old gradients so they dont confuse the new calculations
                loss.backward() #compute new gradients (backpropagation)
                optimizer.step() #updates weights
            total_loss += loss.item() * x.size(0) #average losses * images in the batch = total losses
            correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n

In [32]:
for epoch in range(5):
    tl, ta = run_epoch(train_loader, train=True)
    vl, va = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch+1}: train acc {ta:.3f} | val loss {vl:.3f} | val acc {va:.3f}")

Epoch 1: train acc 0.272 | val loss 3.571 | val acc 0.448
Epoch 2: train acc 0.634 | val loss 2.794 | val acc 0.575
Epoch 3: train acc 0.738 | val loss 2.350 | val acc 0.614
Epoch 4: train acc 0.792 | val loss 2.047 | val acc 0.654
Epoch 5: train acc 0.831 | val loss 1.860 | val acc 0.669


In [42]:
results = {}
results["with_flip"] = {"val_acc": va, "val_loss": vl}
print(results)

{'with_flip': {'val_acc': 0.669431279620853, 'val_loss': 1.8604557898372271}}


This time we'll run everything exactly the same, except with no horizontal flips

In [35]:
# Adjust if your structure differs (e.g. there may be an extra "data" folder)
data_dir = os.path.join(dataset_path, "coins", "data")
train_dir = os.path.join(data_dir, "train")
val_dir   = os.path.join(data_dir, "validation")   # may be "valid"
test_dir  = os.path.join(data_dir, "test")

IMG_SIZE = 224
BATCH_SIZE = 32 #how mant images go through the model at finetuning. How many are we looking at before updating the weights,
#batch size is to look at when were looking at training time and resources

# ImageNet stats - use these if you plan to use a pretrained model
#Noramlise computes (pixel - mean) / std
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

#rotate, flip, and color jitter make the image look a bit different every time to make the model learn what a coin looks like, rrather than what the images look like
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(20),          # coins can be any orientation
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), #converts pixels from 0-255 to 0-1 and rearranged image into a tensor
    transforms.Normalize(mean, std), #shifts and scales the tensor
])

#we don't need random augmentation for the evaluation and testing sets
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
#the transforms are stored in the ImageFolder and applied each time the image is loaded. This ensures the random augmentations are different every epoch
train_ds = datasets.ImageFolder(train_dir, transform=train_transform)
val_ds   = datasets.ImageFolder(val_dir,   transform=eval_transform)
test_ds  = datasets.ImageFolder(test_dir,  transform=eval_transform)

#wraps a dataset and hands it off to the model, armed with all the level settings we decided on
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("Classes:", len(train_ds.classes)) #counts number of classes
print("Train/Val/Test sizes:", len(train_ds), len(val_ds), len(test_ds))

Classes: 211
Train/Val/Test sizes: 6413 844 844


In [36]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(36)

In [37]:
num_classes = len(train_ds.classes)

#resnet is pretrained on imagenet, so it knows to recognize features of an image
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

for p in model.parameters():
    #requires_grad flags each parameter on whether or not to compute gradients for it.
    #were setting it to false for now so that they're excluded from learning to keep pretrained knowledge in tact and save cpu
    #consider playing with it
    p.requires_grad = False          # freeze the pretrained backbone

#model fc (fully connected) is the last layer of resnet, turns the features into class scores
#this is the only part that learns
model.fc = nn.Linear(model.fc.in_features, num_classes)   # new head for 211 coins
model = model.to(device)

#how wrong is the model? 
criterion = nn.CrossEntropyLoss()
#tries to minimize the wrongness
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

In [38]:
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, n = 0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader: #x - images, a tensor shaped [32, 3, 224, 224], y - true labels, tensor or 32 class indices
            x, y = x.to(device), y.to(device) #both sent to device
            out = model(x)
            loss = criterion(out, y) #average loss per image in batch
            if train:
                optimizer.zero_grad() #clear the old gradients so they dont confuse the new calculations
                loss.backward() #compute new gradients (backpropagation)
                optimizer.step() #updates weights
            total_loss += loss.item() * x.size(0) #average losses * images in the batch = total losses
            correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n

for epoch in range(5):
    tl, ta = run_epoch(train_loader, train=True)
    vl, va = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch+1}: train acc {ta:.3f} | val loss {vl:.3f} | val acc {va:.3f}")

Epoch 1: train acc 0.274 | val loss 3.549 | val acc 0.457
Epoch 2: train acc 0.645 | val loss 2.696 | val acc 0.556
Epoch 3: train acc 0.754 | val loss 2.320 | val acc 0.608
Epoch 4: train acc 0.805 | val loss 2.015 | val acc 0.661
Epoch 5: train acc 0.840 | val loss 1.840 | val acc 0.659


In [43]:
results["_flip"] = {"val_acc": va, "val_loss": vl}
print(results)

{'with_flip': {'val_acc': 0.669431279620853, 'val_loss': 1.8604557898372271}, 'no_flip': {'val_acc': 0.6587677725118484, 'val_loss': 1.8402982469983575}}


In [45]:
for p in model.parameters():
    p.requires_grad = True

backbone = [p for n, p in model.named_parameters() if not n.startswith("fc")]
optimizer = torch.optim.Adam([
    {"params": backbone,            "lr": 3e-5},
    {"params": model.fc.parameters(), "lr": 3e-4},
])

import time
for epoch in range(5):
    t0 = time.time()
    tl, ta = run_epoch(train_loader, train=True)
    vl, va = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch+1}: train acc {ta:.3f} | val loss {vl:.3f} | val acc {va:.3f} | {time.time()-t0:.0f}s")
    if epoch == 0 or va > best_va:
        best_va = va
        torch.save(model.state_dict(), "best_coin_model.pt")

Epoch 1: train acc 0.977 | val loss 1.128 | val acc 0.780 | 1674s
Epoch 2: train acc 0.981 | val loss 1.107 | val acc 0.797 | 3101s
Epoch 3: train acc 0.981 | val loss 1.092 | val acc 0.788 | 1901s
Epoch 4: train acc 0.984 | val loss 1.104 | val acc 0.802 | 2116s
Epoch 5: train acc 0.982 | val loss 1.118 | val acc 0.799 | 1931s


In [46]:
torch.save(model.state_dict(), "best_coin_model.pt")

In [47]:
model.eval()
test_loss, test_acc = run_epoch(test_loader, train=False)
print(f"Test loss {test_loss:.3f} | Test acc {test_acc:.3f}")

Test loss 0.873 | Test acc 0.842


In [48]:
with open(os.path.join(data_dir, "cat_to_name.json")) as f:
    cat_to_name = json.load(f)

class_names = [cat_to_name[c] for c in train_ds.classes]
print(len(class_names), class_names[:3])

211 ['1 Cent,Australian dollar,australia', '5 Centavos,Brazilian Real,brazil', '5 Won,Korean Won,south_korea']


In [49]:
from collections import defaultdict, Counter

counts = Counter(train_ds.targets)      # training images per class
correct = defaultdict(int)
total = defaultdict(int)

model.eval()
with torch.no_grad():
    for x, y in val_loader:
        preds = model(x.to(device)).argmax(1).cpu()
        for t, p in zip(y, preds):
            total[t.item()] += 1
            correct[t.item()] += int(t == p)

# 10 worst classes
worst = sorted(total, key=lambda c: correct[c] / total[c])[:10]
for c in worst:
    print(f"{class_names[c]:<40} {correct[c]}/{total[c]}   train imgs: {counts[c]}")

2 euro Cent,Euro,spain                   0/4   train imgs: 28
5 Kopeks,Russian Ruble,russia            1/4   train imgs: 28
5 Dollars,Singapore Dollar,singapore     1/4   train imgs: 8
1 Franc,Swiss Franc,switzerland          1/4   train imgs: 14
2 Francs,Swiss Franc,switzerland         1/4   train imgs: 28
1 Satang,Thai Baht,thailand              1/4   train imgs: 24
5 Satang,Thai Baht,thailand              1/4   train imgs: 24
1 Krone,Danish Krone,denmark             1/4   train imgs: 28
5 Kroner,Danish Krone,denmark            1/4   train imgs: 28
1 Euro,Euro,spain                        1/4   train imgs: 28


In [50]:
low  = [c for c in total if counts[c] <= 15]
rest = [c for c in total if counts[c] > 15]

def acc(classes):
    return sum(correct[c] for c in classes) / sum(total[c] for c in classes)

print(f"Coins with <=15 train images: {len(low)} classes, acc {acc(low):.3f}")
print(f"Coins with  >15 train images: {len(rest)} classes, acc {acc(rest):.3f}")

Coins with <=15 train images: 4 classes, acc 0.625
Coins with  >15 train images: 207 classes, acc 0.802


In [59]:
from PIL import Image
import torch.nn.functional as F

def predict(image_path, top_k=3):
    img = Image.open(image_path).convert("RGB")
    x = eval_transform(img).unsqueeze(0).to(device)    # add batch dimension

    model.eval()
    with torch.no_grad():
        probs = F.softmax(model(x), dim=1)[0]

    top_p, top_i = probs.topk(top_k)
    for p, i in zip(top_p, top_i):
        print(f"{class_names[i]:<40} {p.item()*100:.1f}%")

print("5 shek")
predict(r"C:\Users\USER\Downloads\20261019_195006.jpg")
print("1 shek")
predict(r"C:\Users\USER\Downloads\20261019_194945.jpg")

5 shek
5 Won,Korean Won,south_korea             34.9%
1 Kopek,Russian Ruble,russia             21.8%
50 Kopeks,Russian Ruble,russia           10.8%
1 shek
1 2 New Sheqel,Israeli New Shekel,israel 54.4%
1 New Sheqel,Israeli New Shekel,israel   14.3%
1 Yuan,Chinese Yuan Renminbi,china       9.3%


In [57]:
all_true, all_pred = [], []
model.eval()
with torch.no_grad():
    for x, y in val_loader:
        all_pred += model(x.to(device)).argmax(1).cpu().tolist()
        all_true += y.tolist()

for c in worst:
    wrong = [class_names[p] for t, p in zip(all_true, all_pred) if t == c and p != c]
    print(class_names[c])
    for w in wrong:
        print("   predicted:", w)

2 euro Cent,Euro,spain
   predicted: 2 Euro,Euro,spain
   predicted: 2 Euro,Euro,spain
   predicted: 5 euro Cent,Euro,spain
   predicted: 20 Forint,Hungarian Forint,hungary
5 Kopeks,Russian Ruble,russia
   predicted: 5 Cents,South African Rand,south_africa
   predicted: 1 Kurus,Turkish Lira,turkey
   predicted: 1 Cent,US Dollar,usa
5 Dollars,Singapore Dollar,singapore
   predicted: 10 Kroner,Danish Krone,denmark
   predicted: 10 Dollars,taiwan Dollar,taiwan
   predicted: 5 Jiao,Chinese Yuan Renminbi,china
1 Franc,Swiss Franc,switzerland
   predicted: 2 Francs,Swiss Franc,switzerland
   predicted: 2 Francs,Swiss Franc,switzerland
   predicted: 1 2 Franc,Swiss Franc,switzerland
2 Francs,Swiss Franc,switzerland
   predicted: 1 2 Franc,Swiss Franc,switzerland
   predicted: 10 Rappen,Swiss Franc,switzerland
   predicted: 1 2 Franc,Swiss Franc,switzerland
1 Satang,Thai Baht,thailand
   predicted: 25 Satang,Thai Baht,thailand
   predicted: 1 Baht,Thai Baht,thailand
   predicted: 10 Dollars,ta

I'm not really satisfied with the accuracy of this model. I'm going to try a few others.

In [3]:
import os
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [5]:
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

In [23]:
IMG_SIZE = 64   # 32 is the classic size, but coin details vanish at 32

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_ds = datasets.ImageFolder(train_dir, transform=train_transform)
val_ds   = datasets.ImageFolder(val_dir,   transform=eval_transform)
test_ds  = datasets.ImageFolder(test_dir,  transform=eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [26]:
class LeNet5(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 6, kernel_size=5),   nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(6, 16, kernel_size=5),  nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((5, 5)),     # makes it work at any input size
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 5 * 5, 120), nn.ReLU(),
            nn.Linear(120, 84),         nn.ReLU(),
            nn.Linear(84, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

set_seed(36)
model = LeNet5(num_classes).to(device)
print(sum(p.numel() for p in model.parameters()), "parameters")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

79091 parameters


In [29]:
import time
best_va = 0

for epoch in range(30):
    t0 = time.time()
    tl, ta = run_epoch(train_loader, train=True)
    vl, va = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch+1}: train acc {ta:.3f} | val loss {vl:.3f} | val acc {va:.3f} | {time.time()-t0:.0f}s")

    if va > best_va:
        best_va = va
        torch.save(model.state_dict(), "best_lenet.pt")

Epoch 1: train acc 0.012 | val loss 5.019 | val acc 0.009 | 54s
Epoch 2: train acc 0.030 | val loss 4.790 | val acc 0.034 | 59s
Epoch 3: train acc 0.048 | val loss 4.651 | val acc 0.050 | 57s
Epoch 4: train acc 0.061 | val loss 4.570 | val acc 0.060 | 54s
Epoch 5: train acc 0.073 | val loss 4.485 | val acc 0.073 | 53s
Epoch 6: train acc 0.085 | val loss 4.428 | val acc 0.095 | 54s
Epoch 7: train acc 0.109 | val loss 4.361 | val acc 0.097 | 192s
Epoch 8: train acc 0.121 | val loss 4.313 | val acc 0.115 | 56s
Epoch 9: train acc 0.146 | val loss 4.189 | val acc 0.147 | 56s
Epoch 10: train acc 0.170 | val loss 4.158 | val acc 0.166 | 56s
Epoch 11: train acc 0.192 | val loss 4.043 | val acc 0.187 | 61s
Epoch 12: train acc 0.219 | val loss 3.998 | val acc 0.190 | 66s
Epoch 13: train acc 0.239 | val loss 3.938 | val acc 0.224 | 93s
Epoch 14: train acc 0.262 | val loss 3.871 | val acc 0.239 | 73s
Epoch 15: train acc 0.271 | val loss 3.883 | val acc 0.233 | 65s
Epoch 16: train acc 0.292 | val l

Maybe we can get a bit more accurate?

In [30]:
for epoch in range(30):
    t0 = time.time()
    tl, ta = run_epoch(train_loader, train=True)
    vl, va = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch+31}: train acc {ta:.3f} | val loss {vl:.3f} | val acc {va:.3f} | {time.time()-t0:.0f}s")

    if va > best_va:
        best_va = va
        torch.save(model.state_dict(), "best_lenet.pt")

Epoch 31: train acc 0.484 | val loss 3.936 | val acc 0.364 | 55s
Epoch 32: train acc 0.506 | val loss 3.990 | val acc 0.363 | 56s
Epoch 33: train acc 0.505 | val loss 4.013 | val acc 0.393 | 55s
Epoch 34: train acc 0.510 | val loss 3.991 | val acc 0.395 | 56s
Epoch 35: train acc 0.526 | val loss 4.066 | val acc 0.396 | 56s
Epoch 36: train acc 0.524 | val loss 4.052 | val acc 0.383 | 60s
Epoch 37: train acc 0.540 | val loss 4.065 | val acc 0.409 | 67s


KeyboardInterrupt: 

In [31]:
model.load_state_dict(torch.load("best_lenet.pt"))
model.eval()
print(run_epoch(test_loader, train=False))

(3.792165117806168, 0.4087677725118483)


stopped to avoid overfitting, and try again with some tweaks

In [32]:
class LeNet5(nn.Module):
    def __init__(self, num_classes, p_drop=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 6, kernel_size=5),   nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(6, 16, kernel_size=5),  nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((5, 5)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 5 * 5, 120), nn.ReLU(),
            nn.Linear(120, 84),         nn.ReLU(),
            nn.Dropout(p_drop),                      # new: randomly zeroes 30% of activations in training
            nn.Linear(84, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [33]:
set_seed(36)
model = LeNet5(num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS = 40
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [34]:
import time
best_va = 0     # reset on purpose: this is a new model and a new file

for epoch in range(EPOCHS):
    t0 = time.time()
    tl, ta = run_epoch(train_loader, train=True)
    vl, va = run_epoch(val_loader, train=False)
    scheduler.step()                                  # once per epoch, after training

    lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch+1}: train acc {ta:.3f} | val loss {vl:.3f} | val acc {va:.3f} | lr {lr:.5f} | {time.time()-t0:.0f}s")

    if va > best_va:
        best_va = va
        torch.save(model.state_dict(), "best_lenet_v2.pt")

Epoch 1: train acc 0.011 | val loss 5.088 | val acc 0.008 | lr 0.00100 | 55s
Epoch 2: train acc 0.022 | val loss 4.915 | val acc 0.019 | lr 0.00099 | 55s
Epoch 3: train acc 0.030 | val loss 4.805 | val acc 0.030 | lr 0.00099 | 54s
Epoch 4: train acc 0.038 | val loss 4.720 | val acc 0.037 | lr 0.00098 | 58s
Epoch 5: train acc 0.044 | val loss 4.644 | val acc 0.043 | lr 0.00096 | 190s
Epoch 6: train acc 0.055 | val loss 4.598 | val acc 0.053 | lr 0.00095 | 110s
Epoch 7: train acc 0.068 | val loss 4.450 | val acc 0.090 | lr 0.00093 | 107s
Epoch 8: train acc 0.084 | val loss 4.335 | val acc 0.095 | lr 0.00090 | 107s
Epoch 9: train acc 0.102 | val loss 4.199 | val acc 0.129 | lr 0.00088 | 59s
Epoch 10: train acc 0.121 | val loss 4.120 | val acc 0.133 | lr 0.00085 | 40s
Epoch 11: train acc 0.133 | val loss 4.044 | val acc 0.165 | lr 0.00082 | 42s
Epoch 12: train acc 0.148 | val loss 3.987 | val acc 0.161 | lr 0.00079 | 44s
Epoch 13: train acc 0.165 | val loss 3.896 | val acc 0.200 | lr 0.000

This is worse. We'll try one more time, this time with a wider network, no dropout, little weight decay.

In [36]:
class WideLeNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 5),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 5), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((6, 6)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 6 * 6, 512), nn.ReLU(),
            nn.Linear(512, 256),        nn.ReLU(),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

set_seed(36)
model = WideLeNet(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
best_va = 0   # then train as before, saving to "best_widelenet.pt"

In [37]:
import time
best_va = 0

for epoch in range(40):
    t0 = time.time()
    tl, ta = run_epoch(train_loader, train=True)
    vl, va = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch+1}: train acc {ta:.3f} | val loss {vl:.3f} | val acc {va:.3f} | {time.time()-t0:.0f}s")

    if va > best_va:
        best_va = va
        torch.save(model.state_dict(), "best_widelenet.pt")

Epoch 1: train acc 0.019 | val loss 5.023 | val acc 0.025 | 36s
Epoch 2: train acc 0.049 | val loss 4.732 | val acc 0.041 | 38s
Epoch 3: train acc 0.068 | val loss 4.460 | val acc 0.056 | 43s
Epoch 4: train acc 0.098 | val loss 4.249 | val acc 0.122 | 46s
Epoch 5: train acc 0.131 | val loss 4.188 | val acc 0.148 | 47s
Epoch 6: train acc 0.163 | val loss 4.006 | val acc 0.187 | 47s
Epoch 7: train acc 0.192 | val loss 3.906 | val acc 0.222 | 47s
Epoch 8: train acc 0.227 | val loss 3.804 | val acc 0.227 | 47s
Epoch 9: train acc 0.269 | val loss 3.654 | val acc 0.263 | 47s
Epoch 10: train acc 0.304 | val loss 3.613 | val acc 0.308 | 48s
Epoch 11: train acc 0.334 | val loss 3.639 | val acc 0.302 | 49s
Epoch 12: train acc 0.365 | val loss 3.594 | val acc 0.323 | 48s
Epoch 13: train acc 0.400 | val loss 3.547 | val acc 0.353 | 56s
Epoch 14: train acc 0.440 | val loss 3.614 | val acc 0.366 | 59s
Epoch 15: train acc 0.470 | val loss 3.545 | val acc 0.392 | 61s
Epoch 16: train acc 0.497 | val lo

In [38]:
class LeNet5Orig(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 6, 5),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(6, 16, 5), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((5, 5)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(400, 120), nn.ReLU(),
            nn.Linear(120, 84),  nn.ReLU(),
            nn.Linear(84, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

In [39]:
candidates = {
    "LeNet (original)":     (LeNet5Orig(num_classes),  "best_lenet.pt"),
    "LeNet (dropout+wd)":   (LeNet5(num_classes),      "best_lenet_v2.pt"),
    "WideLeNet":            (WideLeNet(num_classes),   "best_widelenet.pt"),
}

for name, (net, path) in candidates.items():
    net.load_state_dict(torch.load(path, map_location=device))
    model = net.to(device).eval()          # run_epoch uses the global `model`
    vl, va = run_epoch(val_loader, train=False)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{name:<22} val acc {va:.3f} | val loss {vl:.3f} | {n_params:,} params")

LeNet (original)       val acc 0.409 | val loss 4.065 | 79,091 params
LeNet (dropout+wd)     val acc 0.361 | val loss 3.326 | 79,091 params
WideLeNet              val acc 0.537 | val loss 5.451 | 1,419,411 params


Let's try another model now. AlexNet 

In [40]:
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_ds = datasets.ImageFolder(train_dir, transform=train_transform)
val_ds   = datasets.ImageFolder(val_dir,   transform=eval_transform)
test_ds  = datasets.ImageFolder(test_dir,  transform=eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [41]:
set_seed(36)

model = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
print(model.classifier)     # Dropout, Linear, ReLU, Dropout, Linear, ReLU, Linear(4096 -> 1000)

for p in model.features.parameters():
    p.requires_grad = False          # freeze the convolutional part

model.classifier[6] = nn.Linear(4096, num_classes)   # new 211-class output layer
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-4)

Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to C:\Users\USER/.cache\torch\hub\checkpoints\alexnet-owt-7be5be79.pth


100%|████████████████████████| 233M/233M [04:55<00:00, 827kB/s]


Sequential(
  (0): Dropout(p=0.5, inplace=False)
  (1): Linear(in_features=9216, out_features=4096, bias=True)
  (2): ReLU(inplace=True)
  (3): Dropout(p=0.5, inplace=False)
  (4): Linear(in_features=4096, out_features=4096, bias=True)
  (5): ReLU(inplace=True)
  (6): Linear(in_features=4096, out_features=1000, bias=True)
)


In [42]:
import time
best_va = 0

for epoch in range(8):
    t0 = time.time()
    tl, ta = run_epoch(train_loader, train=True)
    vl, va = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch+1}: train acc {ta:.3f} | val loss {vl:.3f} | val acc {va:.3f} | {time.time()-t0:.0f}s")
    if va > best_va:
        best_va = va
        torch.save(model.state_dict(), "best_alexnet.pt")

Epoch 1: train acc 0.208 | val loss 2.721 | val acc 0.402 | 210s
Epoch 2: train acc 0.539 | val loss 1.951 | val acc 0.571 | 21474s
Epoch 3: train acc 0.669 | val loss 1.702 | val acc 0.620 | 182s
Epoch 4: train acc 0.745 | val loss 1.555 | val acc 0.669 | 228s
Epoch 5: train acc 0.797 | val loss 1.442 | val acc 0.681 | 221s
Epoch 6: train acc 0.832 | val loss 1.348 | val acc 0.719 | 224s
Epoch 7: train acc 0.860 | val loss 1.405 | val acc 0.717 | 218s
Epoch 8: train acc 0.875 | val loss 1.399 | val acc 0.725 | 177s


best_va is not reset, we'll only update it if the fine tuning actually beats the frozen

In [43]:
for p in model.parameters():
    p.requires_grad = True

optimizer = torch.optim.Adam([
    {"params": model.features.parameters(),   "lr": 1e-5},
    {"params": model.classifier.parameters(), "lr": 1e-4},
])

for epoch in range(8):
    t0 = time.time()
    tl, ta = run_epoch(train_loader, train=True)
    vl, va = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch+1}: train acc {ta:.3f} | val loss {vl:.3f} | val acc {va:.3f} | {time.time()-t0:.0f}s")
    if va > best_va:
        best_va = va
        torch.save(model.state_dict(), "best_alexnet.pt")

Epoch 1: train acc 0.883 | val loss 1.416 | val acc 0.737 | 317s
Epoch 2: train acc 0.915 | val loss 1.458 | val acc 0.726 | 284s
Epoch 3: train acc 0.924 | val loss 1.400 | val acc 0.761 | 274s
Epoch 4: train acc 0.938 | val loss 1.402 | val acc 0.749 | 268s
Epoch 5: train acc 0.942 | val loss 1.360 | val acc 0.759 | 271s
Epoch 6: train acc 0.949 | val loss 1.407 | val acc 0.775 | 311s
Epoch 7: train acc 0.957 | val loss 1.371 | val acc 0.768 | 305s
Epoch 8: train acc 0.957 | val loss 1.471 | val acc 0.765 | 296s


In [44]:
#two more epochs to try to get val acc a touch higher

for p in model.parameters():
    p.requires_grad = True

optimizer = torch.optim.Adam([
    {"params": model.features.parameters(),   "lr": 1e-5},
    {"params": model.classifier.parameters(), "lr": 1e-4},
])

for epoch in range(2):                 # 2 epochs, watch the time
    t0 = time.time()
    tl, ta = run_epoch(train_loader, train=True)
    vl, va = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch+1}: train acc {ta:.3f} | val loss {vl:.3f} | val acc {va:.3f} | {time.time()-t0:.0f}s")
    if va > best_va:
        best_va = va
        torch.save(model.state_dict(), "best_alexnet.pt")

Epoch 1: train acc 0.954 | val loss 1.597 | val acc 0.756 | 307s
Epoch 2: train acc 0.954 | val loss 1.514 | val acc 0.754 | 279s


In [45]:
model.load_state_dict(torch.load("best_alexnet.pt", map_location=device))
model.eval()
test_loss, test_acc = run_epoch(test_loader, train=False)
print(f"AlexNet test acc {test_acc:.3f}")

AlexNet test acc 0.775


In [46]:
model.load_state_dict(torch.load("best_alexnet.pt", map_location=device))
model.eval()
print("Pretrained AlexNet test:", run_epoch(test_loader, train=False))

Pretrained AlexNet test: (1.1736585110849678, 0.7748815165876777)


AlexNet from scratch

In [47]:
IMG_SIZE = 128

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_ds = datasets.ImageFolder(train_dir, transform=train_transform)
val_ds   = datasets.ImageFolder(val_dir,   transform=eval_transform)
test_ds  = datasets.ImageFolder(test_dir,  transform=eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [48]:
set_seed(36)
model = models.alexnet(weights=None, num_classes=num_classes).to(device)
print(sum(p.numel() for p in model.parameters()), "parameters")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

57868307 parameters


In [49]:
import time
best_va = 0

for epoch in range(30):
    t0 = time.time()
    tl, ta = run_epoch(train_loader, train=True)
    vl, va = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch+1}: train acc {ta:.3f} | val loss {vl:.3f} | val acc {va:.3f} | {time.time()-t0:.0f}s")
    if va > best_va:
        best_va = va
        torch.save(model.state_dict(), "best_alexnet_scratch.pt")

Epoch 1: train acc 0.008 | val loss 5.130 | val acc 0.009 | 151s
Epoch 2: train acc 0.014 | val loss 4.971 | val acc 0.015 | 170s
Epoch 3: train acc 0.030 | val loss 4.839 | val acc 0.019 | 175s
Epoch 4: train acc 0.039 | val loss 4.678 | val acc 0.032 | 216s
Epoch 5: train acc 0.052 | val loss 4.545 | val acc 0.055 | 283s
Epoch 6: train acc 0.070 | val loss 4.364 | val acc 0.077 | 295s
Epoch 7: train acc 0.089 | val loss 4.270 | val acc 0.095 | 308s
Epoch 8: train acc 0.114 | val loss 4.126 | val acc 0.116 | 310s
Epoch 9: train acc 0.134 | val loss 3.990 | val acc 0.147 | 314s
Epoch 10: train acc 0.156 | val loss 3.912 | val acc 0.168 | 317s
Epoch 11: train acc 0.187 | val loss 3.817 | val acc 0.191 | 312s
Epoch 12: train acc 0.208 | val loss 3.724 | val acc 0.217 | 315s
Epoch 13: train acc 0.238 | val loss 3.688 | val acc 0.231 | 316s
Epoch 14: train acc 0.262 | val loss 3.595 | val acc 0.248 | 316s
Epoch 15: train acc 0.291 | val loss 3.434 | val acc 0.300 | 317s
Epoch 16: train acc

In [50]:
model.load_state_dict(torch.load("best_alexnet_scratch.pt", map_location=device))
model.eval()
print("AlexNet scratch test:", run_epoch(test_loader, train=False))

AlexNet scratch test: (3.0399284769573485, 0.514218009478673)


ResNet50 - test accuracy reached 84.2% with validation reaching 78.0%. 
best_coin_model.pt has the weights from training.

Tests
Horizontal flip - not detectable effect
Frozen backbone with full fine-tune - fine-tuning gave a large accuracy jump
Weak coins were not the mainly low-count ones, the low counts were not far from the rest of the database. Coins just tend to look similar. Might be worth trying this experiment over with a larger corpus.
LeNet was the weakest model. 
LeNet original - 0.409 val
LeNet with dropout and weight decay - 0.361 val. Underfitting, but trying to regularizing it hurt it more.
WideLeNet (1.4M parameters) - 0.537 val. The likelihood of it overfitting limited the model.
AlexNet from scratch (128px) - 0.51 val, improving at epoch 30 (but took too long)
AlexNet pretrained (224px) - 0.775 val
ResNet50 pretrained (224px) - 0.780 val, 0.842 test

Pretrained models preformed better - don't be afraid of adapting other people's work. 
Use regularization when you're worried about overfitting, never underfitting.
In the bottom 10 performing coins, 20/31 errors were still within the same country. Larger images and smaller sections could have mitigated this.